# Run Experiment

In [ ]:
import os
from utils.experiments import ExperimentRunner
import utils.constants as constants
import shutil

In [ ]:
# Directories
SCENARIOS_DIR = "scenarios"
EXP_DIR = "experiments_match"
CHROMA_STFT_DIR = "features/chroma_stft_norm2/"
AUDIO_DIR = "Chopin_Mazurkas/wav_22050_mono/Chopin_Op017No4/"

# Data
systems = ['DTW','NOA', 'MATCH', 'OLTW', 'NOA_MONOTONOUS', 'OLTW_GLOBAL']

In [ ]:
# generate EXP_DIR and subdirectories
os.makedirs(EXP_DIR, exist_ok=True)
for system in systems:
    os.makedirs(os.path.join(EXP_DIR, system), exist_ok=True)

In [ ]:
# create symlink from experiments_match/MATCH to experiments/MATCH, only if it doesn't exist already
cwd = os.getcwd()
match_symlink = os.path.join(EXP_DIR, "MATCH")
oltw_symlink = os.path.join(EXP_DIR, "OLTW")

if not os.path.exists(match_symlink):
    os.symlink(os.path.join(cwd, "experiments/MATCH"), match_symlink, target_is_directory=True)
if not os.path.exists(oltw_symlink):
    os.symlink(os.path.join(cwd, "experiments/OLTW"), oltw_symlink, target_is_directory=True)

In [ ]:
# set constants
if EXP_DIR == "experiments_match":
    sr = 44100
    hop_length = 882
    feat_dir = "features/match"
    distance_metric = 'euclidean'
else:
    sr = constants.DEFAULT_SR
    hop_length = constants.DEFAULT_HOP_LENGTH
    feat_dir = CHROMA_STFT_DIR
    distance_metric = 'cosine'

### DTW

In [ ]:
dtw_kwargs = {"steps": constants.DEFAULT_DTW_STEPS,
              "weights": constants.DEFAULT_DTW_WEIGHTS,
              "feat_dir": feat_dir,
              "sr": sr,
              "hop_length": hop_length,
              'distance_metric': distance_metric}
runner = ExperimentRunner("DTW", dtw_kwargs)
runner.run_batch(SCENARIOS_DIR, EXP_DIR)

### NOA

In [ ]:
noa_kwargs = {"steps": constants.DEFAULT_DTW_STEPS,
              "weights": constants.DEFAULT_DTW_WEIGHTS,
              "feat_dir": feat_dir,
              "sr": sr,
              "hop_length": hop_length,
              "norm": True,
              'distance_metric': distance_metric}
runner = ExperimentRunner("NOA", noa_kwargs)
runner.run_batch(SCENARIOS_DIR, EXP_DIR)

#### Monotonous

In [ ]:
noa_kwargs = {"steps": constants.DEFAULT_DTW_STEPS,
              "weights": constants.DEFAULT_DTW_WEIGHTS,
              "feat_dir": feat_dir,
              "sr": sr,
              "hop_length": hop_length,
              "norm": True,
              'distance_metric': distance_metric,
              "monotonous": True}
runner = ExperimentRunner("NOA_MONOTONOUS", noa_kwargs)
runner.run_batch(SCENARIOS_DIR, EXP_DIR)

### MATCH

In [ ]:
assert EXP_DIR != "experiments_match" # avoid running MATCH twice
match_kwargs = {'audio_root': AUDIO_DIR}
runner = ExperimentRunner("MATCH", match_kwargs)
runner.run_batch(SCENARIOS_DIR, EXP_DIR)

### OLTW

In [ ]:
oltw_kwargs = {'hop_length': constants.DEFAULT_HOP_LENGTH}
runner = ExperimentRunner("OLTW", oltw_kwargs)
runner.run_batch(SCENARIOS_DIR, EXP_DIR)

#### Global

In [ ]:
oltw_global_kwargs = {'hop_length': hop_length,
                      'feat_dir': feat_dir,
                      'sr': sr,
                      'distance_metric': distance_metric,
                      'c': 500}
runner = ExperimentRunner("OLTW_GLOBAL", oltw_global_kwargs)
runner.run_batch(SCENARIOS_DIR, EXP_DIR)

### Kalman

In [ ]:
kalman_kwargs = {'feat_dir': CHROMA_STFT_DIR,
                 'sr': constants.DEFAULT_SR,
                 'hop_length': constants.DEFAULT_HOP_LENGTH}
runner = ExperimentRunner("KALMAN", kalman_kwargs)
runner.run_batch(SCENARIOS_DIR, EXP_DIR)

In [ ]:
import numpy as np
np.load("/home/ctang/ttmp/SimRealtimeBenchmark/experiments_match/OLTW_GLOBAL/s24/hyp.npy")